In [ ]:
# Training config — injected at submit time by KaggleTrainingAdapter
CONFIG = {{config}}
print('Experiment:', CONFIG['experiment_name'])
print('Model     :', CONFIG['model'])
print('Epochs    :', CONFIG['epochs'])

In [ ]:
import subprocess, sys

# Install the project package from GitHub (internet must be enabled on the kernel)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'git+{{repo_url}}',
], check=True)

# Install training deps (torch is pre-installed on Kaggle GPU kernels)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'accelerate', 'sentencepiece', 'pydantic',
], check=True)

print('Dependencies installed.')

In [ ]:
import shutil
from pathlib import Path

# Kaggle mounts attached datasets at /kaggle/input/<dataset-slug>/
dataset_slug = CONFIG['experiment_name'] + '-data'
input_base = Path('/kaggle/input') / dataset_slug

# Copy flat .jsonl files from the dataset into data/
data_dst = Path('data')
data_dst.mkdir(exist_ok=True)
for jsonl in input_base.glob('*.jsonl'):
    shutil.copy(jsonl, data_dst / jsonl.name)
    print(f'Copied {jsonl.name} ({jsonl.stat().st_size // 1000} KB)')

In [ ]:
import subprocess, sys

# Package is installed, so cli.train is importable as a module directly
cmd = [
    sys.executable, '-m', 'cli.train',
    '--model', CONFIG['model'],
    '--epochs', str(CONFIG['epochs']),
    '--patience', str(CONFIG['patience']),
    '--warmup-ratio', str(CONFIG['warmup_ratio']),
    '--train-data', 'data/train.jsonl',
    '--eval-data', 'data/eval.jsonl',
    '--output-dir', 'models/checkpoints',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Training complete.')

In [ ]:
import tarfile
from pathlib import Path

checkpoint_dir = Path('models/checkpoints')
output_archive = Path('/kaggle/working/checkpoint.tar.gz')

if not checkpoint_dir.exists():
    raise FileNotFoundError(f'Checkpoint not found at {checkpoint_dir}')

with tarfile.open(output_archive, 'w:gz') as tf:
    tf.add(checkpoint_dir, arcname='checkpoints')

print(f'Packaged -> {output_archive} ({output_archive.stat().st_size / 1e6:.1f} MB)')